# Chronos-2-PETSA Benchmarking

This notebook evaluates the Chronos-2-PETSA (Parameter-Efficient Test-Time Adaptation) model on zero-shot datasets.
It compares the performance (MASE, WQL) and resource usage (Time, Memory) of the standard baseline vs. PETSA.

In [ ]:
# Clone Repository
!git clone https://github.com/Voyagers-time-series-forecasting/Time-Series-Forecasting.git
%cd voyagers-forecasting

In [ ]:
# Install dependencies
!pip install -e .[dev]
!pip install gluonts transformers accelerate typer typer-config rich wandb datasets
!pip install --upgrade sympy

In [ ]:
import sys
import os
import time
import gc
import torch
import wandb
import pandas as pd
from huggingface_hub import HfApi
from google.colab import userdata

sys.path.append(os.path.abspath("src"))

from scripts.evaluation.evaluate import eval_pipeline_and_save_results
from chronos2.pipeline import Chronos2Pipeline
from chronos2.extensions.petsa.petsa import ChronosPETSAWrapper, ChronosPETSAPipeline

In [ ]:
# --- Secrets ---
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    WANDB_KEY = userdata.get('wandb')
    wandb.login(key=WANDB_KEY)
except Exception as e:
    print(f"Could not load secrets: {e}")
    HF_TOKEN = None

# --- Configuration ---
RUN_NAME = "chronos2-petsa-benchmarking"
MODEL_ID = "voyagersnlppolito/chronos2-baseline" 

# Evaluation Config
ZERO_SHOT_CONFIG_PATH = "scripts/evaluation/configs/zero-shot.yaml"

# Standard Config
STANDARD_BATCH_SIZE = 32

# PETSA Config
PETSA_BATCH_SIZE = 1 # Must be 1 for instance-level adaptation
PETSA_RANK = 8
PETSA_ALPHA = 16.0
PETSA_STEPS = 5
PETSA_LR = 1e-3

In [ ]:
class MeasureResources:
    def __init__(self, device="cuda" if torch.cuda.is_available() else "cpu"):
        self.device = device
        self.start_time = 0
        self.end_time = 0
        self.max_memory = 0
        self.duration = 0

    def __enter__(self):
        if self.device == "cuda":
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.empty_cache()
        gc.collect()
        self.start_time = time.time()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end_time = time.time()
        self.duration = self.end_time - self.start_time
        if self.device == "cuda":
            self.max_memory = torch.cuda.max_memory_allocated() / (1024 ** 2) # MB
        else:
            self.max_memory = 0
            
# Initialize WandB Run
if WANDB_KEY:
    wandb.init(project="chronos2-benchmarking", name=RUN_NAME, config={
        "model_id": MODEL_ID,
        "petsa_rank": PETSA_RANK,
        "petsa_steps": PETSA_STEPS,
        "petsa_lr": PETSA_LR
    })

In [ ]:
# Load Pretrained Model
print(f"Loading model: {MODEL_ID}")
pipeline = Chronos2Pipeline.from_pretrained(MODEL_ID, device_map="cuda" if torch.cuda.is_available() else "cpu", torch_dtype=torch.bfloat16)
model = pipeline.model
print("Model loaded successfully.")

In [ ]:
# --- STANDARD EVALUATION ---
print("Starting Standard Zero-Shot Evaluation...")
STANDARD_RESULTS_PATH = "evaluation_results_standard.csv"

if not os.path.exists(ZERO_SHOT_CONFIG_PATH):
    print(f"Config file not found at {ZERO_SHOT_CONFIG_PATH}. Please check the path.")
else:
    with MeasureResources() as standard_res:
        eval_pipeline_and_save_results(
            pipeline=pipeline,
            config_path=ZERO_SHOT_CONFIG_PATH,
            metrics_path=STANDARD_RESULTS_PATH,
            model_id=MODEL_ID,
            batch_size=STANDARD_BATCH_SIZE,
        )
    
    print(f"Standard Eval Duration: {standard_res.duration:.2f} s")
    print(f"Standard Eval Peak Memory: {standard_res.max_memory:.2f} MB")

    # Log to WandB
    if wandb.run is not None:
        try:
            results_df = pd.read_csv(STANDARD_RESULTS_PATH)
            wandb.log({"results/standard": wandb.Table(dataframe=results_df)})
            
            avg_mase = results_df["MASE"].mean()
            avg_wql = results_df["WQL"].mean()
            wandb.log({
                "standard/avg_mase": avg_mase, 
                "standard/avg_wql": avg_wql,
                "standard/time_seconds": standard_res.duration,
                "standard/memory_mb": standard_res.max_memory
            })
        except Exception as e:
            print(f"Failed to log standard results: {e}")

In [ ]:
# --- PETSA WRAPPING ---
print("Wrapping model with PETSA...")
# We wrap the SAME model instance (params will be frozen inside wrapper)
petsa_wrapper = ChronosPETSAWrapper(model, lora_rank=PETSA_RANK, lora_alpha=PETSA_ALPHA)
petsa_pipeline = ChronosPETSAPipeline(petsa_wrapper)
print("PETSA Pipeline ready.")

In [ ]:
# --- PETSA EVALUATION ---
print("Starting PETSA Zero-Shot Evaluation...")
PETSA_RESULTS_PATH = "evaluation_results_petsa.csv"

if not os.path.exists(ZERO_SHOT_CONFIG_PATH):
    print(f"Config file not found at {ZERO_SHOT_CONFIG_PATH}.")
else:
    with MeasureResources() as petsa_res:
        eval_pipeline_and_save_results(
            pipeline=petsa_pipeline,
            config_path=ZERO_SHOT_CONFIG_PATH,
            metrics_path=PETSA_RESULTS_PATH,
            model_id=f"{MODEL_ID}-PETSA",
            batch_size=PETSA_BATCH_SIZE,
        )
        
    print(f"PETSA Eval Duration: {petsa_res.duration:.2f} s")
    print(f"PETSA Eval Peak Memory: {petsa_res.max_memory:.2f} MB")

    # Log to WandB
    if wandb.run is not None:
        try:
            results_df = pd.read_csv(PETSA_RESULTS_PATH)
            wandb.log({"results/petsa": wandb.Table(dataframe=results_df)})
            
            avg_mase = results_df["MASE"].mean()
            avg_wql = results_df["WQL"].mean()
            wandb.log({
                "petsa/avg_mase": avg_mase, 
                "petsa/avg_wql": avg_wql,
                "petsa/time_seconds": petsa_res.duration,
                "petsa/memory_mb": petsa_res.max_memory
            })
        except Exception as e:
            print(f"Failed to log PETSA results: {e}")

In [ ]:
# --- COMPARISON & UPLOAD ---
ratio = petsa_res.duration / standard_res.duration if standard_res.duration > 0 else 0
print(f"\nTime Ratio (PETSA / Standard): {ratio:.2f}x")
if ratio > 2.5:
    print("WARNING: PETSA is taking significantly longer than 2x Standard time.")
    
if HF_TOKEN:
    print("Uploading results to Hub...")
    api = HfApi()
    for path, name in [(STANDARD_RESULTS_PATH, "evaluation_results_standard.csv"), (PETSA_RESULTS_PATH, "evaluation_results_petsa.csv")]:
        if os.path.exists(path):
            try:
                api.upload_file(
                    path_or_fileobj=path,
                    path_in_repo=name,
                    repo_id=f"voyagersnlppolito/{RUN_NAME}",
                    repo_type="model"
                )
                print(f"Uploaded {name}")
            except Exception as e:
                print(f"Failed to upload {name}: {e}")